In [2]:
import sys
sys.path.append("..")  # agar folder src bisa diimpor dari folder notebooks

import pandas as pd
from src.evaluate import muat_semua_hasil, matriks_per_fold, ringkas_per_model

pd.set_option("display.width", 200)

# Gabungkan semua tabel hasil per fold
tabel = pd.read_csv  # (placeholder agar sel ini mudah dibaca; baris berikut yang dipakai)
tabel = muat_semua_hasil("../reports/tables")
print("Jumlah baris hasil:", len(tabel))
print("\nJumlah putaran per model:")
print(tabel.groupby("model").size().to_string())

# Ringkasan: rata-rata dan simpangan baku antar fold (seed dirata-rata dulu)
ringkas = ringkas_per_model(tabel)
print("\nRingkasan semua model (urut Macro F1 tertinggi):")
print(ringkas.round(3).to_string(index=False))
ringkas.round(4).to_csv("../reports/tables/ringkasan_model.csv", index=False)

# Macro F1 per fold untuk tiap model (dirata-rata antar seed)
print("\nMacro F1 per fold:")
print(matriks_per_fold(tabel).round(3).to_string())

Jumlah baris hasil: 95

Jumlah putaran per model:
model
BiLSTM+FastText            15
E3_indobert_lr2e-5         15
E4a_indobert_lr3e-5        15
E4b_indobert_lr1e-5        15
E5_indobert_classweight    15
E6_indobert_slang          15
SVM+TF-IDF                  5

Ringkasan semua model (urut Macro F1 tertinggi):
                  model  macro_f1_mean  macro_f1_std  accuracy_mean  accuracy_std  weighted_f1_mean  weighted_f1_std
    E4b_indobert_lr1e-5          0.856         0.010          0.853         0.010             0.851            0.010
    E4a_indobert_lr3e-5          0.856         0.016          0.852         0.015             0.851            0.016
     E3_indobert_lr2e-5          0.855         0.014          0.852         0.013             0.850            0.013
      E6_indobert_slang          0.853         0.011          0.850         0.011             0.849            0.011
E5_indobert_classweight          0.852         0.013          0.849         0.012             0.847

In [3]:
import numpy as np
from scipy import stats

per_fold = matriks_per_fold(tabel)  # Macro F1 per fold, dirata-rata antar seed
N_LATIH, N_UJI = 1760, 440          # per fold: data latih (train + val) dan data uji

def bandingkan(a, b):
    """Uji berpasangan model a lawan model b pada Macro F1 per fold."""
    x, y = per_fold[a], per_fold[b]
    d = x - y
    k = len(d)
    p_t = stats.ttest_rel(x, y).pvalue
    p_w = stats.wilcoxon(x, y).pvalue  # untuk sampel kecil dihitung secara eksak
    # t-test terkoreksi Nadeau-Bengio: varians dikali (1/k + n_uji/n_latih)
    t_koreksi = d.mean() / np.sqrt((1 / k + N_UJI / N_LATIH) * d.var(ddof=1))
    p_k = 2 * stats.t.sf(abs(t_koreksi), df=k - 1)
    return {"pasangan": f"{a}  vs  {b}",
            "selisih_rata2": d.mean(),
            "fold_menang": f"{int((d > 0).sum())}/{k}",
            "p_t_berpasangan": p_t, "p_wilcoxon": p_w, "p_t_terkoreksi": p_k}

E3, SVM, BILSTM = "E3_indobert_lr2e-5", "SVM+TF-IDF", "BiLSTM+FastText"
pasangan = [(E3, SVM), (E3, BILSTM), (SVM, BILSTM),                # tiga perbandingan utama
            (E3, "E4a_indobert_lr3e-5"), (E3, "E4b_indobert_lr1e-5"),  # variasi vs dasar
            (E3, "E5_indobert_classweight"), (E3, "E6_indobert_slang")]

hasil_uji = pd.DataFrame([bandingkan(a, b) for a, b in pasangan])
print(hasil_uji.round(4).to_string(index=False))
hasil_uji.round(5).to_csv("../reports/tables/uji_signifikansi.csv", index=False)

                                       pasangan  selisih_rata2 fold_menang  p_t_berpasangan  p_wilcoxon  p_t_terkoreksi
             E3_indobert_lr2e-5  vs  SVM+TF-IDF         0.0198         5/5           0.0034      0.0625          0.0141
        E3_indobert_lr2e-5  vs  BiLSTM+FastText         0.0755         5/5           0.0019      0.0625          0.0085
                SVM+TF-IDF  vs  BiLSTM+FastText         0.0557         5/5           0.0057      0.0625          0.0227
    E3_indobert_lr2e-5  vs  E4a_indobert_lr3e-5        -0.0008         2/5           0.8953      1.0000          0.9300
    E3_indobert_lr2e-5  vs  E4b_indobert_lr1e-5        -0.0009         3/5           0.8240      0.8125          0.8819
E3_indobert_lr2e-5  vs  E5_indobert_classweight         0.0033         4/5           0.3820      0.4375          0.5487
      E3_indobert_lr2e-5  vs  E6_indobert_slang         0.0015         3/5           0.6296      0.6250          0.7456
